In [11]:
import pandas as pd
import numpy as np

vehicle_df = pd.read_parquet(r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\bronze\vehicle_insurance\vehicle.parquet")
claims_df = pd.read_parquet(r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\bronze\analytics_vidhya_claims\combined_claims.parquet")

Customer Silver

In [16]:

customer_df = vehicle_df[
    [
        "gender",
        "customer_age",
        "has_driving_license",
        "previously_insured",
        "region"
    ]
]

Generate Keys

In [17]:
customer_df.insert(
    0,
    "customer_sk",
    range(1, len(customer_df) + 1)
)

Age Band

In [18]:
customer_df["age_band"] = pd.cut(
    customer_df["customer_age"],
    bins=[18,25,35,45,60,100],
    labels=["18-25","26-35","36-45","46-60","60+"]
)

Save

In [21]:
import os

file_path = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\silver\customer.csv"

# Ensure the folder directory exists
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Save the file
customer_df.to_csv(file_path, index=False)

# Vehicle Silver

In [22]:
vehicle_df = claims_df.rename(
    columns={
        "age_of_car":"vehicle_age",
        "transmission_type":"transmission",
        "steering_type":"steering"
    }
)

vehicle_df = vehicle_df[
[
"policy_id",
"vehicle_age",
"make",
"model",
"fuel_type",
"segment",
"engine_type",
"displacement",
"cylinder",
"transmission",
"steering",
"length",
"width",
"height",
"gross_weight"
]
]

Generate Vehicle Key

In [23]:
vehicle_df.insert(
    0,
    "vehicle_sk",
    range(1, len(vehicle_df)+1)
)

Save

In [24]:
import os

file_path = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\silver\vehicle.csv"

# Ensure the folder directory exists
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Save the file
customer_df.to_csv(file_path, index=False)

# Safety Table

In [25]:
safety_df = claims_df[
[
"policy_id",
"airbags",
"is_esc",
"is_tpms",
"is_parking_sensors",
"is_parking_camera",
"is_brake_assist",
"is_power_steering",
"is_speed_alert",
"ncap_rating"
]
]

Convert Yes/No

In [27]:
binary_columns = [
"is_esc",
"is_tpms",
"is_parking_sensors",
"is_parking_camera",
"is_brake_assist",
"is_power_steering",
"is_speed_alert"
]

for col in binary_columns:
    safety_df[col] = safety_df[col].map(
        {"Yes":1,"No":0}
    )

Safety Score

In [28]:
safety_df["safety_score"] = (
    safety_df["airbags"]
    + safety_df["is_esc"]
    + safety_df["is_tpms"]
    + safety_df["is_parking_sensors"]
    + safety_df["is_parking_camera"]
    + safety_df["is_brake_assist"]
    + safety_df["is_power_steering"]
    + safety_df["is_speed_alert"]
    + safety_df["ncap_rating"]
)

Save

In [29]:
import os

file_path = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\silver\vehicle_safety.csv"

# Ensure the folder directory exists
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Save the file
customer_df.to_csv(file_path, index=False)

# Create Silver_quote

In [32]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Read source datasets
vehicle_df = pd.read_parquet(
    r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\bronze\vehicle_insurance\vehicle.parquet"
)

# Create Quote Table
quote_df = pd.DataFrame()

# Generate Quote Key
quote_df["quote_sk"] = range(1, len(vehicle_df) + 1)

# Business Key
quote_df["quote_id"] = [f"Q{100000+i}" for i in range(len(vehicle_df))]

# Customer Key
quote_df["customer_sk"] = range(1, len(vehicle_df) + 1)

# Quote Premium (Synthetic)
np.random.seed(42)

# Convert string vehicle age values ('< 1 Year', '1-2 Year', '> 2 Years') to numeric values for calculation
vehicle_age_num = vehicle_df["vehicle_age"].map({
    "< 1 Year": 0.5,
    "1-2 Year": 1.5,
    "> 2 Years": 3.0
}).fillna(1.0)

base_premium = (
    3500
    + vehicle_df["customer_age"] * 20
    + vehicle_age_num * 500
)

quote_df["quoted_premium"] = (
    base_premium + np.random.randint(-500, 500, len(vehicle_df))
).round(0)

# Sales Channel
quote_df["sales_channel"] = vehicle_df["sales_channel"]

# Days Since Last Contact
quote_df["days_since_last_contact"] = vehicle_df["days_since_last_contact"]

# Conversion
quote_df["accepted_offer"] = vehicle_df["accepted_offer"]

# Quote Status
quote_df["quote_status"] = np.where(
    quote_df["accepted_offer"] == 1,
    "Converted",
    "Abandoned"
)

# Quote Stage
quote_df["quote_stage"] = np.where(
    quote_df["accepted_offer"] == 1,
    "Policy Issued",
    np.random.choice(
        [
            "Premium Calculation",
            "Document Upload",
            "Payment",
            "Underwriting"
        ],
        len(vehicle_df)
    )
)

# Device Type
quote_df["device_type"] = np.random.choice(
    ["Mobile", "Desktop", "Tablet"],
    len(vehicle_df),
    p=[0.6, 0.3, 0.1]
)

# Quote Source
quote_df["quote_source"] = np.random.choice(
    ["Website", "Aggregator", "Agent", "Mobile App"],
    len(vehicle_df)
)

# Quote Date
start_date = datetime(2025, 1, 1)

quote_df["quote_date"] = [
    start_date + timedelta(days=int(np.random.randint(0, 365)))
    for _ in range(len(vehicle_df))
]

# Audit Columns
quote_df["load_date"] = pd.Timestamp.now()
quote_df["source_system"] = "Vehicle Insurance"

# Save
quote_df.to_csv(
    r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\silver\quote.csv",
    index=False
)

print("Silver Quote Table Created")

Silver Quote Table Created


Create silver_channel_lookup

In [35]:
import pandas as pd

vehicle_df = pd.read_parquet(
    r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\bronze\vehicle_insurance\vehicle.parquet"
)

# Note: If Policy_Sales_Channel was renamed earlier, use "sales_channel" instead
channels = sorted(vehicle_df["sales_channel"].unique()) if "sales_channel" in vehicle_df.columns else sorted(vehicle_df["Policy_Sales_Channel"].unique())

lookup = pd.DataFrame()

lookup["channel_id"] = channels

lookup["channel_name"] = [
    f"Channel_{i}"
    for i in channels
]

# Fixed: Added 'r' prefix here
lookup.to_csv(
    r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\silver\channel_lookup.csv",
    index=False
)

print(lookup)

     channel_id   channel_name
0           1.0    Channel_1.0
1           2.0    Channel_2.0
2           3.0    Channel_3.0
3           4.0    Channel_4.0
4           6.0    Channel_6.0
..          ...            ...
150       157.0  Channel_157.0
151       158.0  Channel_158.0
152       159.0  Channel_159.0
153       160.0  Channel_160.0
154       163.0  Channel_163.0

[155 rows x 2 columns]


Channel Mapping

In [36]:
channel_mapping = {
    1:"Agent",
    2:"Website",
    3:"Mobile App",
    4:"Branch",
    5:"Aggregator",
    6:"Partner",
    7:"Call Center"
}

lookup["channel_name"] = lookup["channel_id"].map(channel_mapping)

lookup["channel_name"] = lookup["channel_name"].fillna("Other")

# Create silver_policy

In [37]:
import pandas as pd
import numpy as np

# Read Bronze Claims Dataset
claims_df = pd.read_parquet(r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\bronze\analytics_vidhya_claims\combined_claims.parquet")

# Create Policy Table
policy_df = pd.DataFrame()

# Surrogate Key
policy_df["policy_sk"] = range(1, len(claims_df)+1)

# Business Key
policy_df["policy_id"] = claims_df["policy_id"]

# Policy Tenure
policy_df["policy_tenure"] = claims_df["policy_tenure"]

# Policy Tenure Band
policy_df["policy_tenure_band"] = pd.cut(
    policy_df["policy_tenure"],
    bins=[0,0.5,1,5],
    labels=[
        "< 6 Months",
        "6-12 Months",
        "> 12 Months"
    ]
)

# Synthetic Columns
np.random.seed(42)

policy_df["policy_type"] = np.random.choice(
    [
        "Third Party",
        "Comprehensive"
    ],
    len(policy_df),
    p=[0.35,0.65]
)

policy_df["policy_status"] = np.random.choice(
    [
        "Active",
        "Expired",
        "Cancelled"
    ],
    len(policy_df),
    p=[0.90,0.08,0.02]
)

policy_df["coverage_type"] = np.random.choice(
    [
        "Basic",
        "Silver",
        "Gold"
    ],
    len(policy_df)
)

policy_df["load_date"] = pd.Timestamp.now()
policy_df["source_system"] = "Insurance Claims"

# Save
policy_df.to_csv(
    r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\silver\policy.csv",
    index=False
)

print("policy.csv created successfully")

policy.csv created successfully


Create silver_claim

In [38]:
import pandas as pd
import numpy as np

# Read Bronze Dataset
claims_df = pd.read_parquet(r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\bronze\analytics_vidhya_claims\combined_claims.parquet")

# Create Claim Table
claim_df = pd.DataFrame()

# Surrogate Key
claim_df["claim_sk"] = range(1, len(claims_df)+1)

# Business Key
claim_df["claim_id"] = [
    f"CLM{100000+i}"
    for i in range(len(claims_df))
]

claim_df["policy_id"] = claims_df["policy_id"]

# Claim Flag
claim_df["claim_flag"] = claims_df["is_claim"]

# Synthetic Claim Amount
np.random.seed(42)

claim_df["claim_amount"] = np.where(
    claim_df["claim_flag"]==1,
    np.random.randint(
        5000,
        150000,
        len(claim_df)
    ),
    0
)

# Settlement Days
claim_df["settlement_days"] = np.where(
    claim_df["claim_flag"]==1,
    np.random.randint(
        2,
        30,
        len(claim_df)
    ),
    0
)

# Claim Severity
claim_df["claim_severity"] = pd.cut(
    claim_df["claim_amount"],
    bins=[-1,10000,50000,200000],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

# Fraud Risk (Synthetic)
claim_df["fraud_risk"] = np.where(
    claim_df["claim_amount"]>100000,
    "High",
    "Low"
)

claim_df["load_date"] = pd.Timestamp.now()
claim_df["source_system"] = "Insurance Claims"

# Save
claim_df.to_csv(
    r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\silver\claim.csv",
    index=False
)

print("claim.csv created successfully")

claim.csv created successfully


Master Mapping Table 